In [ ]:
import sys

sys.path.append("../")

import math
import os

import ee
import geemap
import geopandas as gpd
import pandas as pd
from dotenv import load_dotenv

import src.dataset.gee_data as gd
import src.dataset.glamos_processing as glamos

In [ ]:
load_dotenv()
gc_project_id = os.getenv("GC_PROJECT_ID")
gd.initialize_gee(gc_project_id)

In [ ]:
def assign_satellite_label(date):
    year = date.year
    if year < 1984:
        return None
    elif 1984 <= year < 2013:
        return "landsat5"
    elif 2013 <= year <= 2016:
        return "landsat8"
    else:
        return "sentinel2"


def prepare_glamos_data(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs(epsg=4326)
    gdf["geometry"] = gdf.geometry.buffer(0)
    gdf["satellite"] = gdf["observation_end"].apply(assign_satellite_label)
    return gdf

In [ ]:
df = pd.read_parquet("../data/processed/glacier_ml_dataset_v1.0.parquet")

In [ ]:
df.describe()

In [ ]:
df.slope_mean.unique()

In [ ]:
df.slope_mean.hist()

In [ ]:
df.aspect_mean.hist()

In [ ]:
df.elev_mean.hist()

In [ ]:
gl = glamos.get_data(2000, 2025)
gl = prepare_glamos_data(gl)

In [ ]:
gl[gl.obs_id == "A10g-18_2008-10-01_2009-09-30"]

In [ ]:
row = gl.iloc[520]
row

In [ ]:
row.geometry

In [ ]:
df[df.obs_id == "A10g-18_2008-10-01_2009-09-30"]

In [ ]:
roi = ee.Geometry(row.geometry.__geo_interface__)

composite = gd.get_glacier_composite(
    sensor_type=row["satellite"],
    polygon=roi,
    start_date=row["observation_start"].strftime("%Y-%m-%d"),
    end_date=row["observation_end"].strftime("%Y-%m-%d"),
    cloud_threshold=40,
)

In [ ]:
composite

In [ ]:
Map = geemap.Map(zoom=12)
mask_params = {"bands": ["mask"], "min": 0.0, "max": 1.0, "gamma": 1.0}
mask2_params = {"bands": ["mask2"], "min": 0.0, "max": 1.0, "gamma": 1.0}
rgb_params = {"bands": ["B4", "B3", "B2"], "min": 0.0, "max": 0.3, "gamma": 1.4}
nir_params = {"bands": ["NIRNEW"], "min": 0.0, "max": 0.3, "gamma": 1.4}

img = composite

ndsi = img.normalizedDifference(["B3", "B11"]).rename("NDSI")
nir = img.select("B8")
red_band = img.select("B4")
swir = img.select("B11")

mask = ndsi.gt(0.4).And(nir.gt(0.11)).rename("mask")
mask2 = ndsi.gt(0.91).And(nir.gt(0.11)).And(red_band.gt(0.2)).rename("mask2")

nir_new = nir.multiply(nir.divide(swir)).rename("NIRNEW")

img = img.addBands([ndsi, mask, mask2, nir_new], overwrite=True)

Map.addLayer(img, nir_params, "Glacier Observation nir")
Map.addLayer(img, mask_params, "Glacier Observation mask")
Map.addLayer(img, mask2_params, "Glacier Observation test")
Map.addLayer(img, rgb_params, "Glacier Observation rgb")

In [ ]:
img.reduceRegion(ee.Reducer.minMax(), roi, scale=30, maxPixels=1e9)

In [ ]:
hist = nir_new.reduceRegion(
    ee.Reducer.histogram(), geometry=roi, scale=30, maxPixels=1e9
).get("NIRNEW")
hist

In [ ]:
counts = ee.Array(ee.Dictionary(hist).get("histogram"))
means = ee.Array(ee.Dictionary(hist).get("bucketMeans"))

In [ ]:
means.length().get([0]).getInfo()

In [ ]:
counts.reduce(ee.Reducer.sum(), [0]).get([0]).getInfo()

In [ ]:
means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0]).getInfo()

In [ ]:
def otsu(hist):
    counts = ee.Array(ee.Dictionary(hist).get("histogram"))
    means = ee.Array(ee.Dictionary(hist).get("bucketMeans"))
    size = means.length().get([0])
    total = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    sum = means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])
    mean = sum.divide(total)

    indices = ee.List.sequence(1, size)

    def calc_bss(i):
        aCounts = counts.slice(0, 0, i)
        aCount = aCounts.reduce(ee.Reducer.sum(), [0]).get([0])
        aMeans = means.slice(0, 0, i)
        aMean = (
            aMeans.multiply(aCounts)
            .reduce(ee.Reducer.sum(), [0])
            .get([0])
            .divide(aCount)
        )

        bCount = total.subtract(aCount)
        bMean = sum.subtract(aCount.multiply(aMean)).divide(bCount)

        return aCount.multiply(aMean.subtract(mean).pow(2)).add(
            bCount.multiply(bMean.subtract(mean).pow(2))
        )

    bss = indices.map(calc_bss)

    return means.sort(bss).get([-1])

In [ ]:
otsu_thresh = otsu(hist)

In [ ]:
snow_mask = nir_new.gte(otsu_thresh).rename("snow_mask")
ice_mask = nir_new.lt(otsu_thresh).rename("ice_mask")

In [ ]:
snow_mask_params = {"bands": ["snow_mask"], "min": 0.0, "max": 1.0, "gamma": 1.0}
ice_mask_params = {"bands": ["ice_mask"], "min": 0.0, "max": 1.0, "gamma": 1.0}

img = img.addBands([snow_mask, ice_mask])

Map.addLayer(img, snow_mask_params, name="Glacier Observation snow")
Map.addLayer(img, ice_mask_params, name="Glacier Observation ice")

In [ ]:
snow_stats = snow_mask.reduceRegion(
    ee.Reducer.sum(), geometry=roi, scale=30, maxPixels=1e9
)
ice_stats = ice_mask.reduceRegion(
    ee.Reducer.sum(), geometry=roi, scale=30, maxPixels=1e9
)

In [ ]:
snow_count = ee.Number(snow_stats.get("snow_mask"))
ice_count = ee.Number(ice_stats.get("ice_mask"))

In [ ]:
total_pixels = snow_count.add(ice_count)
scr = snow_count.divide(total_pixels)
scr

In [ ]:
dem = gd.get_dem(roi).select("DEM")
dem

In [ ]:
dem.reduceRegion(ee.Reducer.minMax(), geometry=roi, scale=30, maxPixels=1e9)

In [ ]:
dem_bins = dem.divide(10).floor().multiply(10).rename("elevation")
dem_bins.reduceRegion(ee.Reducer.minMax(), geometry=roi, scale=30, maxPixels=1e9)

In [ ]:
combined = ee.Image.cat([snow_mask, ice_mask, dem_bins])

In [ ]:
bin_stats = (
    combined.reduceRegion(
        reducer=ee.Reducer.sum().repeat(2).group(groupField=2, groupName="elevation"),
        geometry=roi,
        scale=30,
        maxPixels=1e9,
    )
    .get("groups")
    .getInfo()
)
bin_stats

In [ ]:
bin_stats = sorted(bin_stats, key=lambda x: x["elevation"])
bin_stats

In [ ]:
SCR_SLA_THRESHOLD = 0.6
MIN_CONSECUTIVE_SLA = 3

In [ ]:
def sla(snow_mask: ee.Image, ice_mask: ee.Image, dem: ee.Image, roi: ee.Geometry):
    dem_bins = dem.divide(10).floor().multiply(10).rename("elevation")

    combined = ee.Image.cat([snow_mask, ice_mask, dem_bins])

    bin_stats = (
        combined.reduceRegion(
            reducer=ee.Reducer.sum()
            .repeat(2)
            .group(groupField=2, groupName="elevation"),
            geometry=roi,
            scale=30,
            maxPixels=1e9,
        )
        .get("groups")
        .getInfo()
    )

    if bin_stats is None:
        return None

    bin_stats = sorted(bin_stats, key=lambda x: x["elevation"])

    consecutive_sla_count = 0
    fallback_sla = None
    run_elev = None
    sla = None

    for bin in bin_stats:
        elev = bin["elevation"]
        n_snow = bin["sum"][0]
        n_ice = bin["sum"][1]
        total = n_snow + n_ice

        if total == 0:
            continue

        scr = n_snow / total

        if scr >= SCR_SLA_THRESHOLD:
            print("passed threshold")
            if fallback_sla is None:
                fallback_sla = elev

            if consecutive_sla_count == 0:
                run_elev = elev

            consecutive_sla_count += 1

            if consecutive_sla_count >= MIN_CONSECUTIVE_SLA:
                print("passed 3 count")
                sla = run_elev
                break
        else:
            consecutive_sla_count = 0

    return sla if sla is not None else fallback_sla


In [ ]:
sla_elev = sla(snow_mask, ice_mask, dem, roi)
sla_elev

In [ ]:
sla_mask = dem.gte(sla_elev).rename("sla_mask")

In [ ]:
sla_mask_params = {"bands": ["sla_mask"], "min": 0.0, "max": 1.0, "gamma": 1.0}

img = img.addBands([sla_mask], overwrite=True)

Map.addLayer(img, sla_mask_params, name="Glacier Observation sla")

In [ ]:
def sla_gee(snow_mask: ee.Image, ice_mask: ee.Image, dem: ee.Image, roi: ee.Geometry):
    dem_bins = dem.divide(10).floor().multiply(10).rename("elevation")
    combined = ee.Image.cat([snow_mask, ice_mask, dem_bins])

    stats = combined.reduceRegion(
        reducer=ee.Reducer.sum().repeat(2).group(groupField=2, groupName="elevation"),
        geometry=roi,
        scale=30,
        maxPixels=1e9,
    )

    groups = ee.List(stats.get("groups", ee.List([])))

    def process_bin(b):
        b_dict = ee.Dictionary(b)
        elev = ee.Number(b_dict.get("elevation"))
        counts = ee.List(b_dict.get("sum"))
        snow = ee.Number(counts.get(0))
        ice = ee.Number(counts.get(1))

        total = snow.add(ice)

        scr = ee.Algorithms.If(total.gt(0), snow.divide(total), -1)

        return ee.Feature(None, {"elevation": elev, "scr": scr})

    valid_bins = (
        ee.FeatureCollection(groups.map(process_bin))
        .filter(ee.Filter.gte("scr", 0))
        .sort("elevation")
    )

    elevs = valid_bins.aggregate_array("elevation")
    scrs = valid_bins.aggregate_array("scr")

    mask_list = scrs.map(lambda val: ee.Number(val).gte(SCR_SLA_THRESHOLD))

    fallback_idx = mask_list.indexOf(1)
    fallback = ee.Algorithms.If(fallback_idx.neq(-1), elevs.get(fallback_idx), -9999)

    def find_streak():
        mask_arr = ee.Array(mask_list)
        length = mask_arr.length().get([0])

        a0 = mask_arr.slice(0, 0, length.subtract(2))
        a1 = mask_arr.slice(0, 1, length.subtract(1))
        a2 = mask_arr.slice(0, 2, length)

        consecutive_sums = a0.add(a1).add(a2).toList()

        sla_idx = consecutive_sums.indexOf(3)

        return ee.Algorithms.If(
            sla_idx.neq(-1),
            elevs.get(sla_idx),
            fallback,
        )

    sla = ee.Algorithms.If(mask_list.size().gte(3), find_streak(), fallback)

    return ee.Number(sla)

In [ ]:
sla_gee(snow_mask, ice_mask, dem.select("DEM"), roi).getInfo()

In [ ]:
snow_area_img = snow_mask.selfMask().multiply(ee.Image.pixelArea())
ice_area_img = ice_mask.selfMask().multiply(ee.Image.pixelArea())

snow_stats = snow_area_img.reduceRegion(
    ee.Reducer.sum(), geometry=roi, scale=30, maxPixels=1e9
)
ice_stats = ice_area_img.reduceRegion(
    ee.Reducer.sum(), geometry=roi, scale=30, maxPixels=1e9
)

snow_area = ee.Number(snow_stats.get("snow_mask"))
ice_area = ee.Number(ice_stats.get("ice_mask"))

total_area = snow_area.add(ice_area)
scr = snow_area.divide(total_area)

glacier_area = roi.area()

aar = None
if glacier_area is not None:
    aar = snow_area.divide(glacier_area)

In [ ]:
snow_area.getInfo()

In [ ]:
ice_area.getInfo()

In [ ]:
scr.getInfo()

In [ ]:
aar.getInfo()

In [ ]:
dem = gd.get_dem(roi)
dem_stats = dem.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=roi, scale=30, maxPixels=1e9
)

In [ ]:
dem_stats

In [ ]:
dem_collection = ee.ImageCollection("COPERNICUS/DEM/GLO30")

In [ ]:
dem_collection.size().getInfo()

In [ ]:
dem.select("aspect").reduceRegion(
    reducer=ee.Reducer.mode(), geometry=roi, scale=30, maxPixels=1e9
)

In [ ]:
aspect_int = dem.select("aspect").round()
aspect_int.reduceRegion(
    reducer=ee.Reducer.mode(), geometry=roi, scale=30, maxPixels=1e9
)

In [ ]:
aspect_rad = dem.select("aspect").multiply(math.pi / 180.0)
eastness = aspect_rad.sin().rename("eastness")
northness = aspect_rad.cos().rename("northness")

In [ ]:
combined_img = dem.select(["DEM", "slope", "aspect"]).addBands([eastness, northness])

In [ ]:
mean_stats = combined_img.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=roi, scale=30, maxPixels=1e9
)

In [ ]:
mean_east = ee.Number(mean_stats.get("eastness"))
mean_north = ee.Number(mean_stats.get("northness"))

true_aspect_rad = mean_east.atan2(mean_north)
true_aspect_deg = true_aspect_rad.multiply(180.0 / math.pi)
final_aspect = true_aspect_deg.mod(360)

In [ ]:
final_aspect

In [ ]:
mean_stats.get("aspect")